In [ ]:
# !pip install tabulate
# !pip install ultralytics supervision
# !pip install optuna
# !pip install scikit-learn
# !pip install lightgbm xgboost

# DOWNLOADING VIDEOS INTO THE ENVIRONMENT

In [1]:
import os
import requests
from xml.etree import ElementTree
import pandas as pd
from tqdm import tqdm
import json
import time

def get_complete_bucket_listing(bucket_url):
    """Get complete bucket listing with pagination support"""
    print("🔍 Getting COMPLETE bucket listing...")
    
    all_files = []
    marker = ""
    page = 1
    
    while True:
        print(f"📄 Fetching page {page}...")
        
        if marker:
            url = f"{bucket_url}?marker={marker}"
        else:
            url = bucket_url
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                root = ElementTree.fromstring(response.content)
                
                is_truncated = root.find('{http://doc.s3.amazonaws.com/2006-03-01}IsTruncated')
                is_truncated = is_truncated.text.lower() == 'true' if is_truncated is not None else False
                
                next_marker_elem = root.find('{http://doc.s3.amazonaws.com/2006-03-01}NextMarker')
                next_marker = next_marker_elem.text if next_marker_elem is not None else None
                
                page_files = []
                for content in root.findall('{http://doc.s3.amazonaws.com/2006-03-01}Contents'):
                    key = content.find('{http://doc.s3.amazonaws.com/2006-03-01}Key').text
                    size = int(content.find('{http://doc.s3.amazonaws.com/2006-03-01}Size').text)
                    
                    page_files.append({
                        'original_path': key,  # Keep original path for downloading
                        'target_path': key,    # Default target path
                        'size_bytes': size,
                        'size_mb': size / (1024 * 1024)
                    })
                
                all_files.extend(page_files)
                print(f"   Found {len(page_files)} files on this page")
                
                if is_truncated and next_marker:
                    marker = next_marker
                    page += 1
                else:
                    break
            else:
                print(f"❌ Failed to fetch page {page}: HTTP {response.status_code}")
                break
                
        except Exception as e:
            print(f"❌ Error fetching page {page}: {e}")
            break
    
    print(f"✅ Total files found: {len(all_files):,}")
    return all_files

def organize_files_with_correct_paths(all_files):
    """Organize files with correct download paths but unified local structure"""
    print("\n📁 Organizing files with correct paths...")
    
    camera_files = {
        'normanniles1': [],
        'normanniles2': [], 
        'normanniles3': [],
        'normanniles4': []
    }
    
    for file_info in all_files:
        original_path = file_info['original_path']
        
        # Determine which camera this belongs to and create correct paths
        if original_path.startswith('added/'):
            # Files from added folder: 'added/normanniles1/filename.mp4'
            parts = original_path.split('/')
            if len(parts) >= 3 and parts[1] in camera_files:
                camera = parts[1]
                filename = parts[2]
                
                # Keep original path for downloading, but use simplified path for local storage
                file_info['download_path'] = original_path  # Use original path for download
                file_info['local_path'] = f"{camera}/{filename}"  # Simplified path for local storage
                camera_files[camera].append(file_info)
        
        elif any(original_path.startswith(f"{cam}/") for cam in camera_files.keys()):
            # Files from direct camera folders: 'normanniles1/filename.mp4'
            camera = original_path.split('/')[0]
            filename = original_path.split('/')[1]
            
            if camera in camera_files:
                file_info['download_path'] = original_path  # Use original path for download
                file_info['local_path'] = f"{camera}/{filename}"  # Same for local storage
                camera_files[camera].append(file_info)
    
    # Print organization summary
    for camera, files in camera_files.items():
        print(f"   📹 {camera}: {len(files):,} files")
    
    return camera_files

def create_download_plan(camera_files):
    """Create a comprehensive download plan"""
    print("\n🎯 CREATING DOWNLOAD PLAN")
    print("=" * 50)
    
    total_files = sum(len(files) for files in camera_files.values())
    total_size_gb = sum(sum(f['size_bytes'] for f in files) for files in camera_files.values()) / (1024**3)
    
    print(f"📊 DOWNLOAD SUMMARY:")
    print(f"   Total files: {total_files:,}")
    print(f"   Total size: {total_size_gb:.1f} GB")
    print(f"   Cameras: 4 (normanniles1-4)")
    
    for camera, files in camera_files.items():
        camera_size_gb = sum(f['size_bytes'] for f in files) / (1024**3)
        print(f"   • {camera}: {len(files):,} files, {camera_size_gb:.1f} GB")
    
    return total_files, total_size_gb

def download_camera_files(camera_files, base_download_dir="barbados_traffic"):
    """Download all files with correct URLs but unified local structure"""
    print(f"\n📥 STARTING DOWNLOAD TO: {base_download_dir}")
    print("=" * 60)
    
    os.makedirs(base_download_dir, exist_ok=True)
    
    # Create camera directories
    for camera in camera_files.keys():
        camera_dir = os.path.join(base_download_dir, camera)
        os.makedirs(camera_dir, exist_ok=True)
        print(f"   📁 Created: {camera_dir}")
    
    total_downloaded = 0
    total_size_gb = 0
    failed_downloads = []
    
    # Download files for each camera
    for camera, files in camera_files.items():
        print(f"\n📹 DOWNLOADING {camera.upper()}...")
        print(f"   Files: {len(files):,}")
        
        camera_downloaded = 0
        camera_size_gb = 0
        camera_failed = 0
        
        for file_info in tqdm(files, desc=f"Downloading {camera}"):
            success, file_size = download_single_file_fixed(file_info, base_download_dir)
            if success:
                camera_downloaded += 1
                camera_size_gb += file_info['size_bytes'] / (1024**3)
            else:
                camera_failed += 1
                failed_downloads.append(file_info['download_path'])
        
        total_downloaded += camera_downloaded
        total_size_gb += camera_size_gb
        
        print(f"   ✅ {camera}: {camera_downloaded}/{len(files)} files, {camera_size_gb:.2f} GB")
        if camera_failed > 0:
            print(f"   ❌ {camera}: {camera_failed} failed downloads")
    
    # Report failed downloads
    if failed_downloads:
        print(f"\n⚠️  FAILED DOWNLOADS ({len(failed_downloads)} files):")
        for failed in failed_downloads[:10]:  # Show first 10 failures
            print(f"   • {failed}")
        if len(failed_downloads) > 10:
            print(f"   ... and {len(failed_downloads) - 10} more")
    
    print(f"\n🎉 DOWNLOAD COMPLETED!")
    print(f"   Total: {total_downloaded:,} files, {total_size_gb:.2f} GB")
    print(f"   Failed: {len(failed_downloads):,} files")
    print(f"   Location: {base_download_dir}")
    
    return total_downloaded, total_size_gb, failed_downloads

def download_single_file_fixed(file_info, base_download_dir):
    """Download a single file with correct URL paths"""
    download_path = file_info['download_path']  # Original path from bucket
    local_path = file_info['local_path']        # Simplified local path
    
    file_url = f"https://storage.googleapis.com/brb-traffic/{download_path}"
    
    try:
        full_local_path = os.path.join(base_download_dir, local_path)
        
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(full_local_path), exist_ok=True)
        
        # Skip if file already exists with correct size
        if os.path.exists(full_local_path):
            existing_size = os.path.getsize(full_local_path)
            if existing_size == file_info['size_bytes']:
                return True, file_info['size_bytes']
        
        # Download file
        response = requests.get(file_url, stream=True, timeout=60)
        response.raise_for_status()
        
        # Download with progress for larger files
        total_size = file_info['size_bytes']
        if total_size > 10 * 1024 * 1024:  # Show progress for files > 10MB
            with open(full_local_path, 'wb') as f, tqdm(
                desc=os.path.basename(local_path),
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
                leave=False
            ) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
        else:
            with open(full_local_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        
        # Verify download
        if os.path.getsize(full_local_path) == file_info['size_bytes']:
            return True, file_info['size_bytes']
        else:
            print(f"❌ Size mismatch for {local_path}")
            return False, 0
            
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            print(f"❌ File not found: {download_path}")
        else:
            print(f"❌ HTTP Error {e.response.status_code} for {download_path}")
        return False, 0
    except Exception as e:
        print(f"❌ Failed to download {download_path}: {e}")
        return False, 0

def create_dataset_summary(base_download_dir):
    """Create a summary of the downloaded dataset"""
    print(f"\n📋 CREATING DATASET SUMMARY")
    print("=" * 50)
    
    summary = {
        "total_files": 0,
        "total_size_gb": 0,
        "cameras": {},
        "download_date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    for camera in ['normanniles1', 'normanniles2', 'normanniles3', 'normanniles4']:
        camera_dir = os.path.join(base_download_dir, camera)
        if os.path.exists(camera_dir):
            files = [f for f in os.listdir(camera_dir) if f.endswith('.mp4')]
            total_size = sum(os.path.getsize(os.path.join(camera_dir, f)) for f in files) / (1024**3)
            
            summary["cameras"][camera] = {
                "file_count": len(files),
                "size_gb": round(total_size, 2),
                "files": sorted(files)[:5]  # Sample of first 5 files
            }
            
            summary["total_files"] += len(files)
            summary["total_size_gb"] += total_size
    
    # Save summary
    with open(os.path.join(base_download_dir, "dataset_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    
    print(f"📊 DATASET SUMMARY:")
    print(f"   Total files: {summary['total_files']:,}")
    print(f"   Total size: {summary['total_size_gb']:.2f} GB")
    print(f"   Download date: {summary['download_date']}")
    
    for camera, info in summary["cameras"].items():
        print(f"   • {camera}: {info['file_count']:,} files, {info['size_gb']:.2f} GB")
    
    return summary

# MAIN EXECUTION
if __name__ == "__main__":
    print("🚀 STARTING FIXED BARBADOS TRAFFIC DATASET DOWNLOAD")
    print("=" * 70)
    
    # Step 1: Get complete file listing
    all_files = get_complete_bucket_listing("https://storage.googleapis.com/brb-traffic/")
    
    # Step 2: Organize files with correct paths
    camera_files = organize_files_with_correct_paths(all_files)
    
    # Step 3: Create download plan
    total_files, total_size_gb = create_download_plan(camera_files)
    
    # Step 4: Confirm download
    response = input(f"\n⚠️  Download {total_files:,} files ({total_size_gb:.1f} GB)? (y/n): ")
    if response.lower() != 'y':
        print("Download cancelled.")
        exit()
    
    # Step 5: Download all files
    base_dir = "barbados_traffic"
    downloaded_count, downloaded_size, failed_downloads = download_camera_files(camera_files, base_dir)
    
    # Step 6: Create dataset summary
    summary = create_dataset_summary(base_dir)
    
    print(f"\n✅ DOWNLOAD PROCESS COMPLETED!")
    print(f"   📁 Location: {base_dir}/")
    print(f"   📊 Success: {downloaded_count:,}/{total_files:,} files")
    print(f"   ❌ Failed: {len(failed_downloads):,} files")
    print(f"   💾 Size: {downloaded_size:.2f} GB")
    print(f"   📋 Summary: {base_dir}/dataset_summary.json")

🚀 STARTING FIXED BARBADOS TRAFFIC DATASET DOWNLOAD
🔍 Getting COMPLETE bucket listing...
📄 Fetching page 1...
   Found 1000 files on this page
📄 Fetching page 2...
   Found 1000 files on this page
📄 Fetching page 3...
   Found 1000 files on this page
📄 Fetching page 4...
   Found 1000 files on this page
📄 Fetching page 5...
   Found 1000 files on this page
📄 Fetching page 6...
   Found 1000 files on this page
📄 Fetching page 7...
   Found 1000 files on this page
📄 Fetching page 8...
   Found 1000 files on this page
📄 Fetching page 9...
   Found 1000 files on this page
📄 Fetching page 10...
   Found 1000 files on this page
📄 Fetching page 11...
   Found 1000 files on this page
📄 Fetching page 12...
   Found 1000 files on this page
📄 Fetching page 13...
   Found 1000 files on this page
📄 Fetching page 14...
   Found 1000 files on this page
📄 Fetching page 15...
   Found 1000 files on this page
📄 Fetching page 16...
   Found 1000 files on this page
📄 Fetching page 17...
   Found 1000 files


⚠️  Download 18,716 files (130.9 GB)? (y/n):  y



📥 STARTING DOWNLOAD TO: barbados_traffic
   📁 Created: barbados_traffic/normanniles1
   📁 Created: barbados_traffic/normanniles2
   📁 Created: barbados_traffic/normanniles3
   📁 Created: barbados_traffic/normanniles4

📹 DOWNLOADING NORMANNILES1...
   Files: 4,679


   ✅ normanniles1: 4679/4679 files, 33.58 GB

📹 DOWNLOADING NORMANNILES2...
   Files: 4,679


   ✅ normanniles2: 4679/4679 files, 31.34 GB

📹 DOWNLOADING NORMANNILES3...
   Files: 4,679


   ✅ normanniles3: 4679/4679 files, 33.53 GB

📹 DOWNLOADING NORMANNILES4...
   Files: 4,679


   ✅ normanniles4: 4679/4679 files, 32.44 GB

🎉 DOWNLOAD COMPLETED!
   Total: 18,716 files, 130.89 GB
   Failed: 0 files
   Location: barbados_traffic

📋 CREATING DATASET SUMMARY
📊 DATASET SUMMARY:
   Total files: 18,716
   Total size: 130.89 GB
   Download date: 2026-01-26 22:08:35
   • normanniles1: 4,679 files, 33.58 GB
   • normanniles2: 4,679 files, 31.34 GB
   • normanniles3: 4,679 files, 33.53 GB
   • normanniles4: 4,679 files, 32.44 GB

✅ DOWNLOAD PROCESS COMPLETED!
   📁 Location: barbados_traffic/
   📊 Success: 18,716/18,716 files
   ❌ Failed: 0 files
   💾 Size: 130.89 GB
   📋 Summary: barbados_traffic/dataset_summary.json


### RANDOM ILLUSTRATION HOW OUR YOLO MODEL EXTRACTED SPEED FEATURES FROM THE VIDEOS
FOR NORMANNILE 1,2,4 USE THIS ARE THE ROI = 25%-48% : WHILE FOR NORMANNILE 3 = 20%-42%

In [4]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from collections import deque
import time
import os

# =============================
# CONFIG (Kept the same for consistency)
# =============================
ROI_START_PERCENT = 0.20
ROI_END_PERCENT = 0.42

pixels_per_meter = 50
min_frames_for_direction = 3
size_change_ratio_threshold = 0.03
SIZE_CHANGE_WEIGHT = 5
ENTRY_EXIT_SCORE_THRESHOLD = 4
MIN_FRAMES_FOR_VALID_TRACK = 5

# =============================
# MAIN PIPELINE (Data Collection Phase)
# =============================
def process_video(input_path, output_prefix, confidence_threshold=0.35):

    model = YOLO('yolov10s.pt')
    target_classes = [0, 1, 2, 3, 5, 7]
    class_names = {0:'person', 1:'bicycle', 2:'car',3:'motorcycle',5:'bus',7:'truck'}

    entry_color = (0,255,0)
    exit_color = (0,0,255)
    unknown_color = (0,255,255)
    text_white = (255,255,255)

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("Could not open video.")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    ROI_Y_START = int(height * ROI_START_PERCENT)
    ROI_Y_END   = int(height * ROI_END_PERCENT)

    video_output_path = f"{output_prefix}_video.mp4"
    stats_csv_output_path = f"{output_prefix}_directional_stats.csv"

    out = cv2.VideoWriter(
        video_output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps, (width,height)
    )

    # Tracking & Data Storage
    track_history = {}
    movement_history = {}
    direction_confirmed = {}
    assigned_ids = {}
    next_id = 1
    vehicle_data_points = [] # All raw data points
    density_history = deque(maxlen=60)

    print("Processing video...")

    frame_id = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_id / fps
        results = model.track(
            frame, conf=confidence_threshold,
            classes=target_classes, persist=True, verbose=False
        )

        annotated = frame.copy()
        cv2.line(annotated,(0,ROI_Y_START),(width,ROI_Y_START),(255,0,255),2)
        cv2.line(annotated,(0,ROI_Y_END),(width,ROI_Y_END),(255,0,255),2)

        if results[0].boxes.id is None:
            out.write(annotated)
            frame_id += 1
            continue

        boxes = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        class_ids = results[0].boxes.cls.int().cpu().tolist()

        filtered_ids = []

        # DETECTION LOOP
        for (x,y,w,h), track_id, cls in zip(boxes, track_ids, class_ids):

            if not (ROI_Y_START <= y <= ROI_Y_END):
                continue

            filtered_ids.append(track_id)

            x1,y1 = int(x-w/2), int(y-h/2)
            x2,y2 = int(x+w/2), int(y+h/2)
            area = w*h

            # Assign internal ID (uid)
            if track_id not in assigned_ids:
                assigned_ids[track_id] = {
                    'uid': next_id,
                    'class_id': cls,
                    'initial_pos': (x, y, area),
                    'frame_count': 0,
                    'speeds': deque(maxlen=6),
                    'first_seen_t': t
                }
                next_id += 1

            uid_data = assigned_ids[track_id]
            uid = uid_data['uid']
            cls_name = class_names.get(cls,"obj")
            center = (x,y)
            uid_data['frame_count'] += 1

            speed = 0
            direction = "UNKNOWN"

            if track_id in track_history:
                px,py,pt,pA = track_history[track_id]
                dt = t - pt

                if dt > 0:
                    dist_px = np.linalg.norm(np.array(center) - np.array([px,py]))
                    dist_m = dist_px / pixels_per_meter
                    instant_speed = (dist_m/dt) * 3.6

                    uid_data['speeds'].append(instant_speed)
                    speed = np.mean(uid_data['speeds'])

                    dx,dy = x - px, y - py
                    if track_id not in movement_history: movement_history[track_id] = []
                    movement_history[track_id].append((dx,dy,area))
                    if len(movement_history[track_id]) > 10: movement_history[track_id].pop(0)

                    # Direction scoring logic
                    if len(movement_history[track_id]) >= min_frames_for_direction:
                        mv = movement_history[track_id]
                        avg_x = np.mean([m[0] for m in mv])
                        avg_y = np.mean([m[1] for m in mv])
                        ix,iy,iA = uid_data['initial_pos']
                        size_ratio = (area - iA) / iA if iA > 0 else 0

                        score = 0
                        if abs(avg_y) > 0.5: score += 1 if avg_y > 0 else -1
                        if abs(size_ratio) > size_change_ratio_threshold:
                            score += SIZE_CHANGE_WEIGHT if size_ratio > 0 else -SIZE_CHANGE_WEIGHT

                        if score >= ENTRY_EXIT_SCORE_THRESHOLD: direction="ENTRY"
                        elif score <= -ENTRY_EXIT_SCORE_THRESHOLD: direction="EXIT"

                        if direction != "UNKNOWN":
                            direction_confirmed[track_id] = direction

            if track_id in direction_confirmed:
                direction = direction_confirmed[track_id]

            # LANE MAPPING
            lane = "LEFT" if x < width*0.33 else "MID" if x < width*0.66 else "RIGHT"

            # DATA RECORDING
            vehicle_data_points.append({
                'uid': uid,
                'frame_id': frame_id,
                'class_name': cls_name,
                'speed_kmh': speed,
                'direction': direction
            })

            # Draw box
            color = entry_color if direction=="ENTRY" else exit_color if direction=="EXIT" else unknown_color
            cv2.rectangle(annotated,(x1,y1),(x2,y2),color,2)
            label = f"{cls_name}#{uid} {direction} {speed:.1f}km/h"
            cv2.putText(annotated,label,(x1,y1-8),cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)

            # Update history
            track_history[track_id] = (x,y,t,area)

        density_history.append(len(filtered_ids))

        # Draw Global Panel
        panel = annotated.copy()
        cv2.rectangle(panel,(5,5),(340,130),(0,0,0),-1)
        cv2.addWeighted(panel,0.6,annotated,0.4,0,annotated)
        cv2.putText(annotated,f"Frame {frame_id}",(10,30),0,0.7,(255,255,255),2)
        cv2.putText(annotated,f"Vehicles in ROI: {len(filtered_ids)}",(10,55),0,0.6,(255,255,255),2)
        cv2.putText(annotated,f"Density (avg60f): {np.mean(density_history):.1f}",(10,80),0,0.6,(255,255,255),2)
        cv2.putText(annotated,f"Total Tracks: {len(assigned_ids)}",(10,105),0,0.6,(255,255,255),2)

        out.write(annotated)
        frame_id += 1

    cap.release()
    out.release()

    # ==================================
    # FINAL DATA AGGREGATION AND ANALYSIS
    # ==================================

    # 1. Create DataFrame of per-vehicle statistics
    valid_uids = {v['uid'] for v in assigned_ids.values() if v['frame_count'] >= MIN_FRAMES_FOR_VALID_TRACK}

    final_stats_list = []
    for track_id, uid_data in assigned_ids.items():
        uid = uid_data['uid']
        if uid not in valid_uids:
            continue

        uid_df = pd.DataFrame(vehicle_data_points)[pd.DataFrame(vehicle_data_points)['uid'] == uid]

        # Calculate summary statistics for speed (only using speed > 0)
        speeds = uid_df[uid_df['speed_kmh'] > 0]['speed_kmh']

        speed_avg = speeds.mean() if not speeds.empty else 0

        # Determine final confirmed direction
        final_direction = direction_confirmed.get(track_id, "UNKNOWN")

        final_stats_list.append({
            'uid': uid,
            'direction': final_direction,
            'speed_avg_kmh': speed_avg
        })

    # DataFrame containing the average speed and direction for each vehicle (UID)
    vehicle_summary_df = pd.DataFrame(final_stats_list)

    # 2. Separate data by direction
    entry_speeds = vehicle_summary_df[vehicle_summary_df['direction'] == 'ENTRY']['speed_avg_kmh']
    exit_speeds = vehicle_summary_df[vehicle_summary_df['direction'] == 'EXIT']['speed_avg_kmh']

    # 3. Calculate directional speed statistics
    def calculate_stats(speeds, direction_name):
        if speeds.empty or speeds.sum() == 0:
            return {
                'Direction': direction_name,
                'Count': len(speeds),
                'Max Speed (km/h)': 0.00,
                'Min Speed (km/h)': 0.00,
                'Average Speed (km/h)': 0.00,
                'Speed Range (km/h)': 0.00,
                'Std Deviation (km/h)': 0.00
            }

        max_s = speeds.max()
        min_s = speeds.min()
        avg_s = speeds.mean()
        std_s = speeds.std() if len(speeds) > 1 else 0

        return {
            'Direction': direction_name,
            'Count': len(speeds),
            'Max Speed (km/h)': max_s,
            'Min Speed (km/h)': min_s,
            'Average Speed (km/h)': avg_s,
            'Speed Range (km/h)': max_s - min_s,
            'Std Deviation (km/h)': std_s
        }

    entry_stats = calculate_stats(entry_speeds, "ENTRY")
    exit_stats = calculate_stats(exit_speeds, "EXIT")

    # 4. Create the final required DataFrame
    directional_stats_df = pd.DataFrame([entry_stats, exit_stats])

    # 5. Export and Print Results
    print("\n--- Directional Speed Analysis ---")
    print(directional_stats_df.round(2).to_markdown(index=False))

    directional_stats_df.round(2).to_csv(stats_csv_output_path, index=False)

    print("\nDONE.")
    print(f"Video Output saved: {video_output_path}")
    print(f"Directional Stats CSV saved: {stats_csv_output_path}")


# =============================
# RUN
# =============================
if __name__ == "__main__":
    # Ensure this path is correct for your environment
    inp = "barbados_traffic/normanniles3/normanniles3_2025-10-26-17-53-45.mp4"

    # Use a descriptive prefix for outputs
    output_prefix = f"traffic_analysis_output_{int(time.time())}"

    process_video(inp, output_prefix, confidence_threshold=0.68)

Processing video...

--- Directional Speed Analysis ---
| Direction   |   Count |   Max Speed (km/h) |   Min Speed (km/h) |   Average Speed (km/h) |   Speed Range (km/h) |   Std Deviation (km/h) |
|:------------|--------:|-------------------:|-------------------:|-----------------------:|---------------------:|-----------------------:|
| ENTRY       |      41 |               7.78 |               0.61 |                   3.01 |                 7.17 |                   1.83 |
| EXIT        |      20 |              11.01 |               1.86 |                   6.38 |                 9.15 |                   2.48 |

DONE.
Video Output saved: traffic_analysis_output_1769465721_video.mp4
Directional Stats CSV saved: traffic_analysis_output_1769465721_directional_stats.csv


### FOR ALL VIDEO EXTRACTION THAT WERE IN THE CLEANED TRAIN THIS IS BELOW IS WHAT SHOULD BE DONE

BUT THEN WE COMMENTED ALL THIS OUT BECAUSE IT WOULD HAVE TAKEN 87 HOURS TO FINISH EXTRACTING ALL THIS FOR TRAIN AND THEN LIKE 24 HOURS FOR TEST IN ONE NOTEBOOK.SO INSTEAD OF OVERWHELMING ONE NOTEBOOK ITS BETTER TO SPLIT THE TASKS AND DO IT ON MULTIPLE NOTEBOOKS IN MULTIPLE STUDIOS-MAKING USAGE OF MULTIPLE CORES AND SAVING SO MUCH TIME


In [5]:
# import cv2
# import numpy as np
# import pandas as pd
# from ultralytics import YOLO
# from collections import deque
# import time
# import os
# from tqdm import tqdm 
# import sys
# import threading 

# # =============================
# # CONFIG & CONSTANTS
# # =============================
# # Set this to the root folder where your normanniles folders are located
# VIDEO_ROOT_DIR = "barbados_traffic"
# # Use the threshold you found effective for best results
# CONFIDENCE_THRESHOLD = 0.68
# # This is a critical hyperparameter and should be tuned based on your camera view
# PIXELS_PER_METER = 50

# # Tracking logic constants (keep these for consistent tracking)
# MIN_FRAMES_FOR_DIRECTION = 3
# SIZE_CHANGE_RATIO_THRESHOLD = 0.03
# SIZE_CHANGE_WEIGHT = 5
# ENTRY_EXIT_SCORE_THRESHOLD = 4
# MIN_FRAMES_FOR_VALID_TRACK = 5

# # As per your notes for dynamic ROI
# ROI_MAP = {
#     'Norman Niles #1': (0.25, 0.48), 
#     'Norman Niles #2': (0.25, 0.48),
#     'Norman Niles #3': (0.20, 0.42), 
#     'Norman Niles #4': (0.25, 0.48)
# }

# # Define the structure for the default zero-count DataFrame
# DEFAULT_ZERO_COUNT_DF = lambda time_segment_id, view_label: pd.DataFrame({
#     'time_segment_id': time_segment_id, 'view_label': view_label,
#     'Direction': ['ENTRY', 'EXIT'], 'Vehicle_Count': [0, 0],
#     'Avg_Speed_kmh': [0.00, 0.00], 'Max_Speed_kmh': [0.00, 0.00],
#     'Min_Speed_kmh': [0.00, 0.00], 'Speed_Range_kmh': [0.00, 0.00],
#     'Speed_Std_kmh': [0.00, 0.00]
# })

# # =============================
# # CORE PIPELINE (Data Collection Phase)
# # =============================
# def process_video_data_extraction(
#     input_path, time_segment_id, view_label,
#     roi_start_percent, roi_end_percent,
#     confidence_threshold=CONFIDENCE_THRESHOLD, pixels_per_meter=PIXELS_PER_METER
# ):
#     """
#     Processes a single video to extract directional speed and vehicle count features.
#     """
#     try:
#         model = YOLO('yolov10s.pt')
#     except Exception as e:
#         tqdm.write(f"Error loading YOLO model: {e}")
#         return None

#     target_classes = [0, 1, 2, 3, 5, 7]

#     cap = cv2.VideoCapture(input_path)
#     if not cap.isOpened():
#         return DEFAULT_ZERO_COUNT_DF(time_segment_id, view_label)

#     fps = cap.get(cv2.CAP_PROP_FPS)
#     height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

#     ROI_Y_START = int(height * roi_start_percent)
#     ROI_Y_END = int(height * roi_end_percent)

#     # Tracking & Data Storage Initialization 
#     track_history = {}
#     movement_history = {}
#     direction_confirmed = {}
#     assigned_ids = {}
#     next_id = 1
#     vehicle_data_points = [] 

#     frame_id = 0
#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break

#         t = frame_id / fps
#         results = model.track(
#             frame, conf=confidence_threshold,
#             classes=target_classes, persist=True, verbose=False
#         )

#         if results[0].boxes.id is None:
#             frame_id += 1
#             continue

#         boxes = results[0].boxes.xywh.cpu().numpy()
#         track_ids = results[0].boxes.id.int().cpu().tolist()
#         class_ids = results[0].boxes.cls.int().cpu().tolist()

#         # DETECTION LOOP 
#         for (x,y,w,h), track_id, cls in zip(boxes, track_ids, class_ids):
#             if not (ROI_Y_START <= y <= ROI_Y_END): continue

#             if track_id not in assigned_ids:
#                 assigned_ids[track_id] = {
#                     'uid': next_id, 'class_id': cls, 'initial_pos': (x, y, w*h),
#                     'frame_count': 0, 'speeds': deque(maxlen=6), 'first_seen_t': t
#                 }
#                 next_id += 1

#             uid_data = assigned_ids[track_id]
#             uid = uid_data['uid']
#             center = (x,y)
#             uid_data['frame_count'] += 1

#             speed = 0
#             direction = "UNKNOWN"

#             if track_id in track_history:
#                 px,py,pt,pA = track_history[track_id]
#                 dt = t - pt

#                 if dt > 0:
#                     dist_px = np.linalg.norm(np.array(center) - np.array([px,py]))
#                     dist_m = dist_px / pixels_per_meter
#                     instant_speed = (dist_m/dt) * 3.6

#                     uid_data['speeds'].append(instant_speed)
#                     speed = np.mean(uid_data['speeds'])

#                     dx,dy = x - px, y - py
#                     area = w*h
#                     if track_id not in movement_history: movement_history[track_id] = []
#                     movement_history[track_id].append((dx,dy,area))
#                     if len(movement_history[track_id]) > 10: movement_history[track_id].pop(0)

#                     if len(movement_history[track_id]) >= MIN_FRAMES_FOR_DIRECTION:
#                         mv = movement_history[track_id]
#                         avg_y = np.mean([m[1] for m in mv])
#                         ix,iy,iA = uid_data['initial_pos']
#                         size_ratio = (area - iA) / iA if iA > 0 else 0

#                         score = 0
#                         if abs(avg_y) > 0.5: score += 1 if avg_y > 0 else -1
#                         if abs(size_ratio) > SIZE_CHANGE_RATIO_THRESHOLD:
#                             score += SIZE_CHANGE_WEIGHT if size_ratio > 0 else -SIZE_CHANGE_WEIGHT

#                         if score >= ENTRY_EXIT_SCORE_THRESHOLD: direction="ENTRY"
#                         elif score <= -ENTRY_EXIT_SCORE_THRESHOLD: direction="EXIT"

#                         if direction != "UNKNOWN":
#                             direction_confirmed[track_id] = direction

#             if track_id in direction_confirmed:
#                 direction = direction_confirmed[track_id]

#             vehicle_data_points.append({
#                 'uid': uid, 'frame_id': frame_id, 'class_name': model.names.get(cls, "obj"),
#                 'speed_kmh': speed, 'direction': direction
#             })

#             track_history[track_id] = (x,y,t,w*h)

#         frame_id += 1

#     cap.release()

#     # ==================================
#     # FINAL DATA AGGREGATION AND ANALYSIS
#     # ==================================
#     valid_uids = {v['uid'] for v in assigned_ids.values() if v['frame_count'] >= MIN_FRAMES_FOR_VALID_TRACK}

#     final_stats_list = []
#     vehicle_data_df = pd.DataFrame(vehicle_data_points)

#     for track_id, uid_data in assigned_ids.items():
#         uid = uid_data['uid']
#         if uid not in valid_uids: continue

#         uid_df = vehicle_data_df[vehicle_data_df['uid'] == uid]
#         speeds = uid_df[uid_df['speed_kmh'] > 0]['speed_kmh'] 
#         final_direction = direction_confirmed.get(track_id, "UNKNOWN")

#         if final_direction != "UNKNOWN":
#             final_stats_list.append({
#                 'uid': uid, 'direction': final_direction,
#                 'speed_avg_kmh': speeds.mean() if not speeds.empty else 0
#             })

#     vehicle_summary_df = pd.DataFrame(final_stats_list)

#     # CRITICAL FIX: Handle empty segments
#     if vehicle_summary_df.empty:
#         return DEFAULT_ZERO_COUNT_DF(time_segment_id, view_label)

#     # Calculate directional statistics
#     def calculate_stats(df, direction_name):
#         speeds = df['speed_avg_kmh']
#         count = len(df)
#         return {
#             'time_segment_id': time_segment_id, 'view_label': view_label,
#             'Direction': direction_name,
#             'Vehicle_Count': count,
#             'Avg_Speed_kmh': speeds.mean() if count > 0 else 0.00,
#             'Max_Speed_kmh': speeds.max() if count > 0 else 0.00,
#             'Min_Speed_kmh': speeds.min() if count > 0 else 0.00,
#             'Speed_Range_kmh': speeds.max() - speeds.min() if count > 0 else 0.00,
#             'Speed_Std_kmh': speeds.std() if count > 1 else 0.00
#         }

#     entry_stats = calculate_stats(vehicle_summary_df[vehicle_summary_df['direction'] == 'ENTRY'], "ENTRY")
#     exit_stats = calculate_stats(vehicle_summary_df[vehicle_summary_df['direction'] == 'EXIT'], "EXIT")

#     directional_stats_df = pd.DataFrame([entry_stats, exit_stats])

#     return directional_stats_df.round(2)


# # =============================
# # MAIN EXECUTION SCRIPT
# # =============================
# if __name__ == "__main__":
#     # --- 1. Load Data ---
#     try:
#         cleaned_train = pd.read_csv("cleaned_train.csv")
#         cleaned_train['time_segment_id'] = cleaned_train['time_segment_id'].astype(int)
#         print(f"Loaded {len(cleaned_train)} rows from cleaned_train.csv.")
#     except FileNotFoundError:
#         print("CRITICAL ERROR: 'cleaned_train.csv' not found. Please ensure it is in the current directory.")
#         exit()
#     except Exception as e:
#         print(f"CRITICAL ERROR loading data: {e}. Exiting.")
#         exit()

#     all_features = []
    
#     # --- INITIAL RUN: Process the first 3 segments for confirmation ---
#     cleaned_train_to_process = cleaned_train.head(3) 
    
#     total_segments = len(cleaned_train_to_process)
#     output_filename = "extracted_traffic_features_full.csv"

#     # --- 2. Iterate and Process Initial Videos with TQDM Progress Bar ---
#     print("\nStarting initial confirmation run (First 3 segments)...")
    
#     # Run the loop with tqdm
#     for index, row in tqdm(cleaned_train_to_process.iterrows(), total=total_segments, desc="🚗 Processing Video Segments"):
#         video_path = os.path.join(VIDEO_ROOT_DIR, row['videos'])
#         view = row['view_label']
#         segment_id = row['time_segment_id']
#         video_file = row['videos'] 
        
#         # Update the progress bar description with the current segment and video file
#         tqdm.set_lock(tqdm.get_lock() or threading.Lock())
#         tqdm.get_lock().acquire()
#         tqdm.write(f"  -> Extracting features for segment {segment_id} ({view}) from **{video_file}**...", end='\r')
#         tqdm.get_lock().release()

#         if view not in ROI_MAP:
#             continue
        
#         roi_start, roi_end = ROI_MAP[view]

#         try:
#             segment_features = process_video_data_extraction(
#                 input_path=video_path,
#                 time_segment_id=segment_id,
#                 view_label=view,
#                 roi_start_percent=roi_start,
#                 roi_end_percent=roi_end
#             )
            
#             if segment_features is not None:
#                 # *** ADDED: Attach the video filename to the results ***
#                 segment_features['videos'] = row['videos'] 
#                 all_features.append(segment_features)

#         except Exception as e:
#             tqdm.write(f"CRITICAL ERROR processing segment {segment_id} ({view}): {e}. Skipping.", file=sys.stderr)
#             continue
    
#     # --- 3. Confirmation and Interactive Prompt ---
#     if all_features:
#         confirmation_df = pd.concat(all_features, ignore_index=True)
        
#         print("\n" + "="*80)
#         print("✅ **INITIAL RUN COMPLETE (First 3 Segments)**")
#         print("Total feature rows created:", len(confirmation_df))
#         print("-" * 80)
        
#         # Display the results neatly - NOW SHOWING ALL COLUMNS
#         pd.set_option('display.max_columns', None)
#         pd.set_option('display.width', 1000)
#         print(confirmation_df.to_string())
        
#         print("-" * 80)
        
#         # Interactive prompt
#         while True:
#             response = input("Is the initial result OK? Press 'y' to continue with the full dataset (9014 segments) and save to CSV, or 'n' to stop: ").lower()
            
#             # Reset display options after showing the frame
#             pd.reset_option('display.max_columns')
#             pd.reset_option('display.width')

#             if response == 'y':
#                 all_features = []
#                 cleaned_train_to_process = cleaned_train 
#                 total_segments = len(cleaned_train_to_process)
#                 print("\nStarting **FULL** dataset processing...")
                
#                 # Rerun the loop for the FULL dataset
#                 for index, row in tqdm(cleaned_train_to_process.iterrows(), total=total_segments, desc="🚗 Processing Full Dataset"):
#                     video_path = os.path.join(VIDEO_ROOT_DIR, row['videos'])
#                     view = row['view_label']
#                     segment_id = row['time_segment_id']
#                     video_file = row['videos'] 

#                     tqdm.set_lock(tqdm.get_lock() or threading.Lock())
#                     tqdm.get_lock().acquire()
#                     tqdm.write(f"  -> Extracting features for segment {segment_id} ({view}) from **{video_file}**...", end='\r')
#                     tqdm.get_lock().release()

#                     if view not in ROI_MAP:
#                         continue
                    
#                     roi_start, roi_end = ROI_MAP[view]

#                     try:
#                         segment_features = process_video_data_extraction(
#                             input_path=video_path,
#                             time_segment_id=segment_id,
#                             view_label=view,
#                             roi_start_percent=roi_start,
#                             roi_end_percent=roi_end
#                         )
                        
#                         if segment_features is not None:
#                             # *** ADDED: Attach the video filename to the results ***
#                             segment_features['videos'] = row['videos'] 
#                             all_features.append(segment_features)

#                     except Exception as e:
#                         tqdm.write(f"CRITICAL ERROR processing segment {segment_id} ({view}): {e}. Skipping.", file=sys.stderr)
#                         continue
                
#                 # FINAL SAVE after full run
#                 if all_features:
#                     final_features_df = pd.concat(all_features, ignore_index=True)
                    
#                     # Create the unique ID
#                     final_features_df['ID'] = (
#                         final_features_df['time_segment_id'].astype(str) + '_' +
#                         final_features_df['view_label'].str.replace(' ', '_') + '_' +
#                         final_features_df['Direction'].str.lower() + '_rating'
#                     )

#                     # Select and reorder final columns, including 'videos'
#                     final_features_df = final_features_df[[
#                         'ID', 'time_segment_id', 'view_label', 'Direction', 'videos', # <--- 'videos' column included here
#                         'Vehicle_Count', 'Avg_Speed_kmh', 'Max_Speed_kmh',
#                         'Min_Speed_kmh', 'Speed_Range_kmh', 'Speed_Std_kmh'
#                     ]]

#                     final_features_df.to_csv(output_filename, index=False)
                    
#                     print("\n" + "═"*80)
#                     print(f"🎉 **FULL RUN COMPLETE!**")
#                     print(f"✅ All {len(final_features_df)} feature rows saved to: **{output_filename}**")
#                     print("═"*80)
#                 else:
#                     print("\n❌ Full run completed, but no features were extracted.")
                
#                 break 
                
#             elif response == 'n':
#                 print("Stopping script as requested. No CSV file was saved.")
#                 break 
#             else:
#                 print("Invalid input. Please enter 'y' or 'n'.")

#     else:
#         print("\n❌ Initial run failed to extract any features. Cannot proceed.")

Loaded 9014 rows from cleaned_train.csv.

Starting initial confirmation run (First 3 segments)...


🚗 Processing Video Segments:   0%|          | 0/3 [00:00<?, ?it/s]

🚗 Processing Video Segments:  33%|███▎      | 1/3 [03:35<07:11, 215.84s/it]

🚗 Processing Video Segments:  67%|██████▋   | 2/3 [07:12<03:36, 216.44s/it]

🚗 Processing Video Segments: 100%|██████████| 3/3 [10:56<00:00, 218.86s/it]



✅ **INITIAL RUN COMPLETE (First 3 Segments)**
Total feature rows created: 6
--------------------------------------------------------------------------------
   time_segment_id       view_label Direction  Vehicle_Count  Avg_Speed_kmh  Max_Speed_kmh  Min_Speed_kmh  Speed_Range_kmh  Speed_Std_kmh                                             videos
0                5  Norman Niles #1     ENTRY              1           6.30           6.30           6.30             0.00           0.00  normanniles1/normanniles1_2025-10-20-06-05-45.mp4
1                5  Norman Niles #1      EXIT              0           0.00           0.00           0.00             0.00           0.00  normanniles1/normanniles1_2025-10-20-06-05-45.mp4
2                6  Norman Niles #1     ENTRY              4           6.63           7.92           5.11             2.81           1.39  normanniles1/normanniles1_2025-10-20-06-06-45.mp4
3                6  Norman Niles #1      EXIT              1           5.84           

Is the initial result OK? Press 'y' to continue with the full dataset (9014 segments) and save to CSV, or 'n' to stop:  n


Stopping script as requested. No CSV file was saved.


## TOOK AROUND 24-27 HOURS FOR ALL WORK TO BE FINISHED

### SPLITTING TRAIN INTO 4 EQUAL PORTIONS IN EACH FOLDER

In [4]:
import pandas as pd
import numpy as np
import os

# --- Configuration ---
INPUT_FILE = "cleaned_train.csv"
OUTPUT_DIR = "split_traffic_data_train"
NOTEBOOKS_PER_FOLDER = 4  # The number of processes (notebooks) you will run per folder

# --- Execution ---
if __name__ == "__main__":
    
    # 1. Load the Master CSV
    try:
        df_master = pd.read_csv(INPUT_FILE)
        df_master['time_segment_id'] = df_master['time_segment_id'].astype(int)
        print(f"Loaded {len(df_master)} total rows from {INPUT_FILE}.")
    except FileNotFoundError:
        print(f"CRITICAL ERROR: {INPUT_FILE} not found.")
        import sys
        sys.exit()
    
    # Create the output directory if it doesn't exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Group the data by view_label (Norman Niles #1, #2, #3, #4)
    grouped_data = df_master.groupby('view_label')
    
    total_files_created = 0
    
    print("\nStarting data splitting process...")
    
    # 2. Iterate through each unique folder/view
    for view_label, df_group in grouped_data:
        num_segments = len(df_group)
        print(f"  -> Processing '{view_label}': {num_segments} segments total.")
        
        # Calculate indices for splitting
        split_indices = np.linspace(0, num_segments, NOTEBOOKS_PER_FOLDER + 1, dtype=int)
        
        # 3. Iterate through the chunks (the 4 notebooks per folder)
        for i in range(NOTEBOOKS_PER_FOLDER):
            start_idx = split_indices[i]
            end_idx = split_indices[i + 1]
            
            # Skip if the chunk would be empty
            if start_idx >= end_idx:
                continue
                
            # Get the chunk
            df_chunk = df_group.iloc[start_idx:end_idx]
            
            # Create a unique filename: e.g., Norman_Niles_1_Part_1.csv
            safe_view_label = view_label.replace(' ', '_').replace('#', '')
            chunk_number = i + 1
            output_filename = os.path.join(
                OUTPUT_DIR, 
                f"{safe_view_label}_Part_{chunk_number}.csv"
            )
            
            # Save the chunk to a new CSV
            df_chunk.to_csv(output_filename, index=False)
            total_files_created += 1
            
            print(f"    - Saved Part {chunk_number} ({len(df_chunk)} rows) to: {output_filename}")

    # 4. Final Summary
    print("\n" + "="*60)
    print(f"🎉 SUCCESS! Data splitting complete.")
    print(f"Total files created: {total_files_created} (4 folders * 4 parts = 16)")
    print(f"Output files are located in the **{OUTPUT_DIR}** directory.")
    print("="*60)

Loaded 9014 total rows from cleaned_train.csv.

Starting data splitting process...
  -> Processing 'Norman Niles #1': 1155 segments total.
    - Saved Part 1 (288 rows) to: split_traffic_data_train/Norman_Niles_1_Part_1.csv
    - Saved Part 2 (289 rows) to: split_traffic_data_train/Norman_Niles_1_Part_2.csv
    - Saved Part 3 (289 rows) to: split_traffic_data_train/Norman_Niles_1_Part_3.csv
    - Saved Part 4 (289 rows) to: split_traffic_data_train/Norman_Niles_1_Part_4.csv
  -> Processing 'Norman Niles #2': 2857 segments total.
    - Saved Part 1 (714 rows) to: split_traffic_data_train/Norman_Niles_2_Part_1.csv
    - Saved Part 2 (714 rows) to: split_traffic_data_train/Norman_Niles_2_Part_2.csv
    - Saved Part 3 (714 rows) to: split_traffic_data_train/Norman_Niles_2_Part_3.csv
    - Saved Part 4 (715 rows) to: split_traffic_data_train/Norman_Niles_2_Part_4.csv
  -> Processing 'Norman Niles #3': 2699 segments total.
    - Saved Part 1 (674 rows) to: split_traffic_data_train/Norman_Nil

### STARTED EXTRACTING TRAIN

In [ ]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from collections import deque
import time
import os
from tqdm import tqdm 
import sys
import threading 

# ===============================================
# ⚠️ USER CONFIGURATION: CHANGE THESE TWO LINES FOR EACH JOB ⚠️
# ===============================================
INPUT_FILE_FOR_THIS_JOB = "split_traffic_data_train/Norman_Niles_1_Part_1.csv" # <-- 1 of 16 CSVs
PROCESS_ID = "A1_N1"                                                     # <-- Unique ID (e.g., A1_N1, A2_N3, etc.)

# ===============================================
# CONFIG & CONSTANTS (KEEP THESE AS THEY ARE)
# ===============================================
VIDEO_ROOT_DIR = "barbados_traffic"
CONFIDENCE_THRESHOLD = 0.68
PIXELS_PER_METER = 50

# Tracking logic constants
MIN_FRAMES_FOR_DIRECTION = 3
SIZE_CHANGE_RATIO_THRESHOLD = 0.03
SIZE_CHANGE_WEIGHT = 5
ENTRY_EXIT_SCORE_THRESHOLD = 4
MIN_FRAMES_FOR_VALID_TRACK = 5

# Dynamic ROI Map
ROI_MAP = {
    'Norman Niles #1': (0.25, 0.48), 
    'Norman Niles #2': (0.25, 0.48),
    'Norman Niles #3': (0.20, 0.42), 
    'Norman Niles #4': (0.25, 0.48)
}

# Default Zero Count DataFrame function
DEFAULT_ZERO_COUNT_DF = lambda time_segment_id, view_label: pd.DataFrame({
    'time_segment_id': time_segment_id, 'view_label': view_label,
    'Direction': ['ENTRY', 'EXIT'], 'Vehicle_Count': [0, 0],
    'Avg_Speed_kmh': [0.00, 0.00], 'Max_Speed_kmh': [0.00, 0.00],
    'Min_Speed_kmh': [0.00, 0.00], 'Speed_Range_kmh': [0.00, 0.00],
    'Speed_Std_kmh': [0.00, 0.00]
})

# =============================
# CORE PIPELINE (Data Collection Phase) - UNCHANGED
# =============================
def process_video_data_extraction(
    input_path, time_segment_id, view_label,
    roi_start_percent, roi_end_percent,
    confidence_threshold=CONFIDENCE_THRESHOLD, pixels_per_meter=PIXELS_PER_METER
):
    try:
        # Load the model here (or pass it in if pre-loading saves time on your specific machine)
        model = YOLO('yolov10s.pt') 
    except Exception as e:
        tqdm.write(f"Error loading YOLO model: {e}")
        return None

    target_classes = [0, 1, 2, 3, 5, 7]
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        return DEFAULT_ZERO_COUNT_DF(time_segment_id, view_label)

    fps = cap.get(cv2.CAP_PROP_FPS)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ROI_Y_START = int(height * roi_start_percent)
    ROI_Y_END = int(height * roi_end_percent)

    track_history = {}
    movement_history = {}
    direction_confirmed = {}
    assigned_ids = {}
    next_id = 1
    vehicle_data_points = [] 

    frame_id = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_id / fps
        results = model.track(frame, conf=confidence_threshold, classes=target_classes, persist=True, verbose=False)

        if results[0].boxes.id is None:
            frame_id += 1
            continue

        boxes = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        class_ids = results[0].boxes.cls.int().cpu().tolist()

        for (x,y,w,h), track_id, cls in zip(boxes, track_ids, class_ids):
            if not (ROI_Y_START <= y <= ROI_Y_END): continue

            if track_id not in assigned_ids:
                assigned_ids[track_id] = {'uid': next_id, 'class_id': cls, 'initial_pos': (x, y, w*h), 'frame_count': 0, 'speeds': deque(maxlen=6), 'first_seen_t': t}
                next_id += 1

            uid_data = assigned_ids[track_id]
            uid = uid_data['uid']
            center = (x,y)
            uid_data['frame_count'] += 1

            speed = 0
            direction = "UNKNOWN"

            if track_id in track_history:
                px,py,pt,pA = track_history[track_id]
                dt = t - pt

                if dt > 0:
                    dist_px = np.linalg.norm(np.array(center) - np.array([px,py]))
                    dist_m = dist_px / pixels_per_meter
                    instant_speed = (dist_m/dt) * 3.6

                    uid_data['speeds'].append(instant_speed)
                    speed = np.mean(uid_data['speeds'])

                    dx,dy = x - px, y - py
                    area = w*h
                    if track_id not in movement_history: movement_history[track_id] = []
                    movement_history[track_id].append((dx,dy,area))
                    if len(movement_history[track_id]) > 10: movement_history[track_id].pop(0)

                    if len(movement_history[track_id]) >= MIN_FRAMES_FOR_DIRECTION:
                        mv = movement_history[track_id]
                        avg_y = np.mean([m[1] for m in mv])
                        ix,iy,iA = uid_data['initial_pos']
                        size_ratio = (area - iA) / iA if iA > 0 else 0

                        score = 0
                        if abs(avg_y) > 0.5: score += 1 if avg_y > 0 else -1
                        if abs(size_ratio) > SIZE_CHANGE_RATIO_THRESHOLD:
                            score += SIZE_CHANGE_WEIGHT if size_ratio > 0 else -SIZE_CHANGE_WEIGHT

                        if score >= ENTRY_EXIT_SCORE_THRESHOLD: direction="ENTRY"
                        elif score <= -ENTRY_EXIT_SCORE_THRESHOLD: direction="EXIT"

                        if direction != "UNKNOWN":
                            direction_confirmed[track_id] = direction

            if track_id in direction_confirmed:
                direction = direction_confirmed[track_id]

            vehicle_data_points.append({'uid': uid, 'frame_id': frame_id, 'class_name': model.names.get(cls, "obj"), 'speed_kmh': speed, 'direction': direction})
            track_history[track_id] = (x,y,t,w*h)

        frame_id += 1
    cap.release()

    valid_uids = {v['uid'] for v in assigned_ids.values() if v['frame_count'] >= MIN_FRAMES_FOR_VALID_TRACK}
    final_stats_list = []
    vehicle_data_df = pd.DataFrame(vehicle_data_points)

    for track_id, uid_data in assigned_ids.items():
        uid = uid_data['uid']
        if uid not in valid_uids: continue

        uid_df = vehicle_data_df[vehicle_data_df['uid'] == uid]
        speeds = uid_df[uid_df['speed_kmh'] > 0]['speed_kmh'] 
        final_direction = direction_confirmed.get(track_id, "UNKNOWN")

        if final_direction != "UNKNOWN":
            final_stats_list.append({'uid': uid, 'direction': final_direction, 'speed_avg_kmh': speeds.mean() if not speeds.empty else 0})

    vehicle_summary_df = pd.DataFrame(final_stats_list)

    if vehicle_summary_df.empty: return DEFAULT_ZERO_COUNT_DF(time_segment_id, view_label)

    def calculate_stats(df, direction_name):
        speeds = df['speed_avg_kmh']
        count = len(df)
        return {
            'time_segment_id': time_segment_id, 'view_label': view_label, 'Direction': direction_name, 'Vehicle_Count': count,
            'Avg_Speed_kmh': speeds.mean() if count > 0 else 0.00, 'Max_Speed_kmh': speeds.max() if count > 0 else 0.00,
            'Min_Speed_kmh': speeds.min() if count > 0 else 0.00, 'Speed_Range_kmh': speeds.max() - speeds.min() if count > 0 else 0.00,
            'Speed_Std_kmh': speeds.std() if count > 1 else 0.00
        }

    entry_stats = calculate_stats(vehicle_summary_df[vehicle_summary_df['direction'] == 'ENTRY'], "ENTRY")
    exit_stats = calculate_stats(vehicle_summary_df[vehicle_summary_df['direction'] == 'EXIT'], "EXIT")
    directional_stats_df = pd.DataFrame([entry_stats, exit_stats])
    return directional_stats_df.round(2)
    
# =============================
# MAIN EXECUTION SCRIPT
# =============================
if __name__ == "__main__":
    
    # 1. Load the specific job file
    output_filename = f"extracted_traffic_features_{PROCESS_ID}.csv"
    
    try:
        # Load the pre-split CSV for this job
        cleaned_train_to_process = pd.read_csv(INPUT_FILE_FOR_THIS_JOB)
        cleaned_train_to_process['time_segment_id'] = cleaned_train_to_process['time_segment_id'].astype(int)
        
        total_segments_in_job = len(cleaned_train_to_process)
        
        print(f"Loaded {total_segments_in_job} rows for Job **{PROCESS_ID}** from: {INPUT_FILE_FOR_THIS_JOB}")
        print(f"Results will be saved to: **{output_filename}**")

    except FileNotFoundError:
        print(f"CRITICAL ERROR: Input file '{INPUT_FILE_FOR_THIS_JOB}' not found. Check if the 'split_traffic_data' directory exists.")
        sys.exit()
    except Exception as e:
        print(f"CRITICAL ERROR loading data: {e}. Exiting.")
        sys.exit()

    all_features = []

    # 2. Iterate and Process Videos with TQDM Progress Bar
    for index, row in tqdm(cleaned_train_to_process.iterrows(), total=total_segments_in_job, desc=f"🚗 Job {PROCESS_ID} Progress"):
        video_path = os.path.join(VIDEO_ROOT_DIR, row['videos'])
        view = row['view_label']
        segment_id = row['time_segment_id']
        video_file = row['videos'] 

        # Log the current segment being processed cleanly above the progress bar
        tqdm.set_lock(tqdm.get_lock() or threading.Lock())
        tqdm.get_lock().acquire()
        tqdm.write(f"  -> Extracting features for segment {segment_id} ({view}) from **{video_file}**...", end='\r')
        tqdm.get_lock().release()

        if view not in ROI_MAP: continue
        
        roi_start, roi_end = ROI_MAP[view]

        try:
            segment_features = process_video_data_extraction(
                input_path=video_path, time_segment_id=segment_id, view_label=view,
                roi_start_percent=roi_start, roi_end_percent=roi_end
            )
            
            if segment_features is not None:
                segment_features['videos'] = row['videos'] 
                all_features.append(segment_features)

        except Exception as e:
            tqdm.write(f"CRITICAL ERROR processing segment {segment_id} ({view}): {e}. Skipping.", file=sys.stderr)
            continue
    
    # 3. Final Save for this process
    if all_features:
        final_features_df = pd.concat(all_features, ignore_index=True)
        
        final_features_df['ID'] = (
            final_features_df['time_segment_id'].astype(str) + '_' +
            final_features_df['view_label'].str.replace(' ', '_') + '_' +
            final_features_df['Direction'].str.lower() + '_rating'
        )

        final_features_df = final_features_df[[
            'ID', 'time_segment_id', 'view_label', 'Direction', 'videos',
            'Vehicle_Count', 'Avg_Speed_kmh', 'Max_Speed_kmh',
            'Min_Speed_kmh', 'Speed_Range_kmh', 'Speed_Std_kmh'
        ]]

        final_features_df.to_csv(output_filename, index=False)
        
        print("\n" + "═"*80)
        print(f"🎉 **JOB {PROCESS_ID} COMPLETE!**")
        print(f"✅ Extracted {len(final_features_df)} feature rows and saved to: **{output_filename}**")
        print("═"*80)
    else:
        print("\n❌ Job completed, but no features were successfully extracted.")

Loaded 288 rows for Job **A1_N1** from: split_traffic_data_train/Norman_Niles_1_Part_1.csv
Results will be saved to: **extracted_traffic_features_A1_N1.csv**


🚗 Job A1_N1 Progress:   0%|          | 0/288 [00:00<?, ?it/s]

### SPLITTING DATA FOR TEST

In [1]:
import pandas as pd
import numpy as np
import os

# --- Configuration ---
INPUT_FILE = "TestInputSegments.csv"
OUTPUT_DIR = "split_traffic_data_test"
NOTEBOOKS_PER_FOLDER = 4  # The number of processes (notebooks) you will run per folder

# --- Execution ---
if __name__ == "__main__":
    
    # 1. Load the Master CSV
    try:
        df_master = pd.read_csv(INPUT_FILE)
        df_master['time_segment_id'] = df_master['time_segment_id'].astype(int)
        print(f"Loaded {len(df_master)} total rows from {INPUT_FILE}.")
    except FileNotFoundError:
        print(f"CRITICAL ERROR: {INPUT_FILE} not found.")
        import sys
        sys.exit()
    
    # Create the output directory if it doesn't exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Group the data by view_label (Norman Niles #1, #2, #3, #4)
    grouped_data = df_master.groupby('view_label')
    
    total_files_created = 0
    
    print("\nStarting data splitting process...")
    
    # 2. Iterate through each unique folder/view
    for view_label, df_group in grouped_data:
        num_segments = len(df_group)
        print(f"  -> Processing '{view_label}': {num_segments} segments total.")
        
        # Calculate indices for splitting
        split_indices = np.linspace(0, num_segments, NOTEBOOKS_PER_FOLDER + 1, dtype=int)
        
        # 3. Iterate through the chunks (the 4 notebooks per folder)
        for i in range(NOTEBOOKS_PER_FOLDER):
            start_idx = split_indices[i]
            end_idx = split_indices[i + 1]
            
            # Skip if the chunk would be empty
            if start_idx >= end_idx:
                continue
                
            # Get the chunk
            df_chunk = df_group.iloc[start_idx:end_idx]
            
            # Create a unique filename: e.g., Norman_Niles_1_Part_1.csv
            safe_view_label = view_label.replace(' ', '_').replace('#', '')
            chunk_number = i + 1
            output_filename = os.path.join(
                OUTPUT_DIR, 
                f"{safe_view_label}_Part_{chunk_number}.csv"
            )
            
            # Save the chunk to a new CSV
            df_chunk.to_csv(output_filename, index=False)
            total_files_created += 1
            
            print(f"    - Saved Part {chunk_number} ({len(df_chunk)} rows) to: {output_filename}")

    # 4. Final Summary
    print("\n" + "="*60)
    print(f"🎉 SUCCESS! Data splitting complete.")
    print(f"Total files created: {total_files_created} (4 folders * 4 parts = 16)")
    print(f"Output files are located in the **{OUTPUT_DIR}** directory.")
    print("="*60)

Loaded 2640 total rows from TestInputSegments.csv.

Starting data splitting process...
  -> Processing 'Norman Niles #1': 660 segments total.
    - Saved Part 1 (165 rows) to: split_traffic_data_test/Norman_Niles_1_Part_1.csv
    - Saved Part 2 (165 rows) to: split_traffic_data_test/Norman_Niles_1_Part_2.csv
    - Saved Part 3 (165 rows) to: split_traffic_data_test/Norman_Niles_1_Part_3.csv
    - Saved Part 4 (165 rows) to: split_traffic_data_test/Norman_Niles_1_Part_4.csv
  -> Processing 'Norman Niles #2': 660 segments total.
    - Saved Part 1 (165 rows) to: split_traffic_data_test/Norman_Niles_2_Part_1.csv
    - Saved Part 2 (165 rows) to: split_traffic_data_test/Norman_Niles_2_Part_2.csv
    - Saved Part 3 (165 rows) to: split_traffic_data_test/Norman_Niles_2_Part_3.csv
    - Saved Part 4 (165 rows) to: split_traffic_data_test/Norman_Niles_2_Part_4.csv
  -> Processing 'Norman Niles #3': 660 segments total.
    - Saved Part 1 (165 rows) to: split_traffic_data_test/Norman_Niles_3_Par

### EXTRACTING FOR TEST

In [ ]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from collections import deque
import time
import os
from tqdm import tqdm 
import sys
import threading 

# ===============================================
# ⚠️ USER CONFIGURATION: CHANGE THESE TWO LINES FOR EACH JOB ⚠️
# ===============================================
INPUT_FILE_FOR_THIS_JOB = "split_traffic_data_test/Norman_Niles_1_Part_1.csv" # <-- 1 of 16 CSVs
PROCESS_ID = "B1_N1"                                                     # <-- Unique ID (e.g., B1_N1, B2_N3, etc.)

# ===============================================
# CONFIG & CONSTANTS (KEEP THESE AS THEY ARE)
# ===============================================
VIDEO_ROOT_DIR = "barbados_traffic"
CONFIDENCE_THRESHOLD = 0.68
PIXELS_PER_METER = 50

# Tracking logic constants
MIN_FRAMES_FOR_DIRECTION = 3
SIZE_CHANGE_RATIO_THRESHOLD = 0.03
SIZE_CHANGE_WEIGHT = 5
ENTRY_EXIT_SCORE_THRESHOLD = 4
MIN_FRAMES_FOR_VALID_TRACK = 5

# Dynamic ROI Map
ROI_MAP = {
    'Norman Niles #1': (0.25, 0.48), 
    'Norman Niles #2': (0.25, 0.48),
    'Norman Niles #3': (0.20, 0.42), 
    'Norman Niles #4': (0.25, 0.48)
}

# Default Zero Count DataFrame function
DEFAULT_ZERO_COUNT_DF = lambda time_segment_id, view_label: pd.DataFrame({
    'time_segment_id': time_segment_id, 'view_label': view_label,
    'Direction': ['ENTRY', 'EXIT'], 'Vehicle_Count': [0, 0],
    'Avg_Speed_kmh': [0.00, 0.00], 'Max_Speed_kmh': [0.00, 0.00],
    'Min_Speed_kmh': [0.00, 0.00], 'Speed_Range_kmh': [0.00, 0.00],
    'Speed_Std_kmh': [0.00, 0.00]
})

# =============================
# CORE PIPELINE (Data Collection Phase) - UNCHANGED
# =============================
def process_video_data_extraction(
    input_path, time_segment_id, view_label,
    roi_start_percent, roi_end_percent,
    confidence_threshold=CONFIDENCE_THRESHOLD, pixels_per_meter=PIXELS_PER_METER
):
    try:
        # Load the model here (or pass it in if pre-loading saves time on your specific machine)
        model = YOLO('yolov10s.pt') 
    except Exception as e:
        tqdm.write(f"Error loading YOLO model: {e}")
        return None

    target_classes = [0, 1, 2, 3, 5, 7]
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        return DEFAULT_ZERO_COUNT_DF(time_segment_id, view_label)

    fps = cap.get(cv2.CAP_PROP_FPS)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ROI_Y_START = int(height * roi_start_percent)
    ROI_Y_END = int(height * roi_end_percent)

    track_history = {}
    movement_history = {}
    direction_confirmed = {}
    assigned_ids = {}
    next_id = 1
    vehicle_data_points = [] 

    frame_id = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_id / fps
        results = model.track(frame, conf=confidence_threshold, classes=target_classes, persist=True, verbose=False)

        if results[0].boxes.id is None:
            frame_id += 1
            continue

        boxes = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        class_ids = results[0].boxes.cls.int().cpu().tolist()

        for (x,y,w,h), track_id, cls in zip(boxes, track_ids, class_ids):
            if not (ROI_Y_START <= y <= ROI_Y_END): continue

            if track_id not in assigned_ids:
                assigned_ids[track_id] = {'uid': next_id, 'class_id': cls, 'initial_pos': (x, y, w*h), 'frame_count': 0, 'speeds': deque(maxlen=6), 'first_seen_t': t}
                next_id += 1

            uid_data = assigned_ids[track_id]
            uid = uid_data['uid']
            center = (x,y)
            uid_data['frame_count'] += 1

            speed = 0
            direction = "UNKNOWN"

            if track_id in track_history:
                px,py,pt,pA = track_history[track_id]
                dt = t - pt

                if dt > 0:
                    dist_px = np.linalg.norm(np.array(center) - np.array([px,py]))
                    dist_m = dist_px / pixels_per_meter
                    instant_speed = (dist_m/dt) * 3.6

                    uid_data['speeds'].append(instant_speed)
                    speed = np.mean(uid_data['speeds'])

                    dx,dy = x - px, y - py
                    area = w*h
                    if track_id not in movement_history: movement_history[track_id] = []
                    movement_history[track_id].append((dx,dy,area))
                    if len(movement_history[track_id]) > 10: movement_history[track_id].pop(0)

                    if len(movement_history[track_id]) >= MIN_FRAMES_FOR_DIRECTION:
                        mv = movement_history[track_id]
                        avg_y = np.mean([m[1] for m in mv])
                        ix,iy,iA = uid_data['initial_pos']
                        size_ratio = (area - iA) / iA if iA > 0 else 0

                        score = 0
                        if abs(avg_y) > 0.5: score += 1 if avg_y > 0 else -1
                        if abs(size_ratio) > SIZE_CHANGE_RATIO_THRESHOLD:
                            score += SIZE_CHANGE_WEIGHT if size_ratio > 0 else -SIZE_CHANGE_WEIGHT

                        if score >= ENTRY_EXIT_SCORE_THRESHOLD: direction="ENTRY"
                        elif score <= -ENTRY_EXIT_SCORE_THRESHOLD: direction="EXIT"

                        if direction != "UNKNOWN":
                            direction_confirmed[track_id] = direction

            if track_id in direction_confirmed:
                direction = direction_confirmed[track_id]

            vehicle_data_points.append({'uid': uid, 'frame_id': frame_id, 'class_name': model.names.get(cls, "obj"), 'speed_kmh': speed, 'direction': direction})
            track_history[track_id] = (x,y,t,w*h)

        frame_id += 1
    cap.release()

    valid_uids = {v['uid'] for v in assigned_ids.values() if v['frame_count'] >= MIN_FRAMES_FOR_VALID_TRACK}
    final_stats_list = []
    vehicle_data_df = pd.DataFrame(vehicle_data_points)

    for track_id, uid_data in assigned_ids.items():
        uid = uid_data['uid']
        if uid not in valid_uids: continue

        uid_df = vehicle_data_df[vehicle_data_df['uid'] == uid]
        speeds = uid_df[uid_df['speed_kmh'] > 0]['speed_kmh'] 
        final_direction = direction_confirmed.get(track_id, "UNKNOWN")

        if final_direction != "UNKNOWN":
            final_stats_list.append({'uid': uid, 'direction': final_direction, 'speed_avg_kmh': speeds.mean() if not speeds.empty else 0})

    vehicle_summary_df = pd.DataFrame(final_stats_list)

    if vehicle_summary_df.empty: return DEFAULT_ZERO_COUNT_DF(time_segment_id, view_label)

    def calculate_stats(df, direction_name):
        speeds = df['speed_avg_kmh']
        count = len(df)
        return {
            'time_segment_id': time_segment_id, 'view_label': view_label, 'Direction': direction_name, 'Vehicle_Count': count,
            'Avg_Speed_kmh': speeds.mean() if count > 0 else 0.00, 'Max_Speed_kmh': speeds.max() if count > 0 else 0.00,
            'Min_Speed_kmh': speeds.min() if count > 0 else 0.00, 'Speed_Range_kmh': speeds.max() - speeds.min() if count > 0 else 0.00,
            'Speed_Std_kmh': speeds.std() if count > 1 else 0.00
        }

    entry_stats = calculate_stats(vehicle_summary_df[vehicle_summary_df['direction'] == 'ENTRY'], "ENTRY")
    exit_stats = calculate_stats(vehicle_summary_df[vehicle_summary_df['direction'] == 'EXIT'], "EXIT")
    directional_stats_df = pd.DataFrame([entry_stats, exit_stats])
    return directional_stats_df.round(2)
    
# =============================
# MAIN EXECUTION SCRIPT
# =============================
if __name__ == "__main__":
    
    # 1. Load the specific job file
    output_filename = f"extracted_traffic_features_{PROCESS_ID}.csv"
    
    try:
        # Load the pre-split CSV for this job
        cleaned_train_to_process = pd.read_csv(INPUT_FILE_FOR_THIS_JOB)
        cleaned_train_to_process['time_segment_id'] = cleaned_train_to_process['time_segment_id'].astype(int)
        
        total_segments_in_job = len(cleaned_train_to_process)
        
        print(f"Loaded {total_segments_in_job} rows for Job **{PROCESS_ID}** from: {INPUT_FILE_FOR_THIS_JOB}")
        print(f"Results will be saved to: **{output_filename}**")

    except FileNotFoundError:
        print(f"CRITICAL ERROR: Input file '{INPUT_FILE_FOR_THIS_JOB}' not found. Check if the 'split_traffic_data' directory exists.")
        sys.exit()
    except Exception as e:
        print(f"CRITICAL ERROR loading data: {e}. Exiting.")
        sys.exit()

    all_features = []

    # 2. Iterate and Process Videos with TQDM Progress Bar
    for index, row in tqdm(cleaned_train_to_process.iterrows(), total=total_segments_in_job, desc=f"🚗 Job {PROCESS_ID} Progress"):
        video_path = os.path.join(VIDEO_ROOT_DIR, row['videos'])
        view = row['view_label']
        segment_id = row['time_segment_id']
        video_file = row['videos'] 

        # Log the current segment being processed cleanly above the progress bar
        tqdm.set_lock(tqdm.get_lock() or threading.Lock())
        tqdm.get_lock().acquire()
        tqdm.write(f"  -> Extracting features for segment {segment_id} ({view}) from **{video_file}**...", end='\r')
        tqdm.get_lock().release()

        if view not in ROI_MAP: continue
        
        roi_start, roi_end = ROI_MAP[view]

        try:
            segment_features = process_video_data_extraction(
                input_path=video_path, time_segment_id=segment_id, view_label=view,
                roi_start_percent=roi_start, roi_end_percent=roi_end
            )
            
            if segment_features is not None:
                segment_features['videos'] = row['videos'] 
                all_features.append(segment_features)

        except Exception as e:
            tqdm.write(f"CRITICAL ERROR processing segment {segment_id} ({view}): {e}. Skipping.", file=sys.stderr)
            continue
    
    # 3. Final Save for this process
    if all_features:
        final_features_df = pd.concat(all_features, ignore_index=True)
        
        final_features_df['ID'] = (
            final_features_df['time_segment_id'].astype(str) + '_' +
            final_features_df['view_label'].str.replace(' ', '_') + '_' +
            final_features_df['Direction'].str.lower() + '_rating'
        )

        final_features_df = final_features_df[[
            'ID', 'time_segment_id', 'view_label', 'Direction', 'videos',
            'Vehicle_Count', 'Avg_Speed_kmh', 'Max_Speed_kmh',
            'Min_Speed_kmh', 'Speed_Range_kmh', 'Speed_Std_kmh'
        ]]

        final_features_df.to_csv(output_filename, index=False)
        
        print("\n" + "═"*80)
        print(f"🎉 **JOB {PROCESS_ID} COMPLETE!**")
        print(f"✅ Extracted {len(final_features_df)} feature rows and saved to: **{output_filename}**")
        print("═"*80)
    else:
        print("\n❌ Job completed, but no features were successfully extracted.")

Loaded 165 rows for Job **B1_N1** from: split_traffic_data_test/Norman_Niles_1_Part_1.csv
Results will be saved to: **extracted_traffic_features_B1_N1.csv**


🚗 Job B1_N1 Progress:   0%|          | 0/165 [00:00<?, ?it/s]

# DO NOT UNDERGO THE PAIN FOR WAITING LIKE WE DID IN THE ZIP YOU WILL FIND EXTRACTED FOLDER HAS ALL CSVS FROM THIS PROCESSES